#Ingesta de carpeta con archivos SCV

In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "production_company"
v_esquema = "movie_silver"
v_tabla = "productions_companies"
v_partition = "file_date"
v_merge_condition = "target.company_id = source.company_id"

In [0]:
#1. Leer archivos CSV usando DataFrameReader de Spark

# Define el la estructura personName
production_company_schema = StructType(fields = [
    StructField("companyId", IntegerType(), True),
    StructField("companyName", StringType(), True)
])

# Cargamos el archivo utilizando la estructura definida
production_company_df = spark.read\
    .schema(production_company_schema)\
    .csv(f"{bronze_folder_path}/{v_file_date}/{v_archivo}")

# Mostramos el resultado
display(production_company_df)

In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
from pyspark.sql.functions import col, concat, current_timestamp, lit
production_company_renamed_df = production_company_df\
    .withColumnRenamed("companyId", "company_id")\
    .withColumnRenamed("companyName", "company_name")

production_company_renamed_df = add_ingestion_date(production_company_renamed_df)
production_company_renamed_df = add_env(production_company_renamed_df)
final_df = add_file_date (production_company_renamed_df)

display(final_df)


In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
resultado = merge_delta_lake (v_esquema, v_tabla, final_df, v_merge_condition, v_partition)
print(resultado)

In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 

#production_company_renamed_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.productions_companies")

#production_company_renamed_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
#print(f"Se insertaron {production_company_renamed_df.count()} registros en la tabla {v_esquema}.{v_tabla}")



In [0]:
dbutils.notebook.exit("El notebook 09. Ingestion folder_production_company, termino correctamente")